## Representation-aware revision

The native VSB audit is diagnostic only. Each raw signal has 800,000 samples at 40 MHz, so the revised experiment evaluates predeclared end-aligned windows and preserves `id_measurement` at the parent-signal level. The grouped holdout and the unlabeled official test are excluded from candidate selection.

# VSB data audit

**Objective.** Inspect the VSB Parquet schema, metadata, labels, signal relationships and measurement grouping.

**Inputs.** `engineering-vsb-power-line-fault-detection@2018-kaggle-snapshot` from the local raw-data adapter.

**Outputs.** Parquet schema, signal-to-metadata alignment, class counts, group composition and a leakage-safe split candidate.

**Experimental role.** Resolve the dataset-specific signal input policy before training.

**Leakage constraints.** Keep every phase of an `id_measurement` together. The official unlabeled test set is not a scientific test target.

In [1]:
from partial_discharge_adaptive_fusion.dataset import audit_vsb_metadata, audit_vsb_signal_sample, inspect_vsb_parquet_schema, load_vsb_metadata, resolve_dataset
from partial_discharge_adaptive_fusion.splits import vsb_grouped_manifest

dataset = resolve_dataset('engineering-vsb-power-line-fault-detection', '2018-kaggle-snapshot')
metadata = load_vsb_metadata(dataset)
print(audit_vsb_metadata(metadata))
print(inspect_vsb_parquet_schema(dataset))
print(audit_vsb_signal_sample(dataset, metadata, sample_size=3))
manifest = vsb_grouped_manifest(metadata, dataset_id=dataset.dataset_id, dataset_version=dataset.version)
manifest.validate()
display(manifest.frame.groupby(['split','label']).size().rename('n').reset_index())

{'n_signals': 8712, 'n_measurements': 2904, 'phase_values': ['0', '1', '2'], 'groups_with_three_distinct_phases': 2904, 'groups_with_unexpected_phase_count': 0, 'groups_with_unexpected_row_count': 0, 'label_counts': {'0': 8187, '1': 525}, 'groups_with_mixed_labels': 38, 'mixed_label_groups': ['1068', '1076', '1091', '1132', '1256', '126', '1268', '1277', '1304', '1420', '1537', '1561', '159', '1668', '1704', '1884', '1899', '1994', '2328', '2623', '2693', '271', '2753', '2760', '2807', '2876', '301', '443', '518', '601', '608', '620', '67', '706', '894', '944', '96', '988']}
{'path_name': 'train.parquet', 'num_rows': 800000, 'num_columns': 8712, 'num_row_groups': 1, 'columns': [{'name': '0', 'type': 'int8', 'nullable': True}, {'name': '1', 'type': 'int8', 'nullable': True}, {'name': '2', 'type': 'int8', 'nullable': True}, {'name': '3', 'type': 'int8', 'nullable': True}, {'name': '4', 'type': 'int8', 'nullable': True}, {'name': '5', 'type': 'int8', 'nullable': True}, {'name': '6', 'type

,split,label,n
0,test,0,1634
1,test,1,106
2,train,0,4916
3,train,1,313
4,validation,0,1637
5,validation,1,106


## Findings and handoff

The audit must record the actual Parquet column names, signal dtype and native signal geometry before setting the VSB temporal/CWT input policy. If that policy cannot be justified without test results, stop.

**Next stage:** update the frozen protocol only after the audit is complete.

## V4 protocol note — [PROPOSED · SCIENTIFIC]

The canonical runtime data source is repository-local `data/raw`, resolved through `PD_RAW_DATA_ROOT`. MATLAB retains 400-sample parent signals. VSB retains 800,000-sample parent signals, three phases per `id_measurement`, and deterministic end-aligned unpadded windows. Window labels are never independent examples; splits and supervision remain at parent-signal level. VSB development selection must not open the grouped holdout or the unlabeled official test.
